# Alfacore V8 - Configurazione Google Drive (Fase 1)
Esegui questa cella per connettere Google Drive e trasferire automaticamente tutti gli script della Fase 1 (Data Pipeline) che abbiamo creato.

In [ ]:
from google.colab import drive
import os

# 1. Monta Google Drive
drive.mount('/content/drive')

# 2. Crea la struttura delle directory sul tuo Drive
drive_path = '/content/drive/MyDrive/alfacore_v8/data_pipeline'
os.makedirs(drive_path, exist_ok=True)
print(f"✅ Directory creata su Google Drive: {drive_path}")

## Iniezione degli Script Python
Le celle seguenti scriveranno fisicamente i file Python all'interno del tuo Google Drive.

In [ ]:
%%writefile /content/drive/MyDrive/alfacore_v8/data_pipeline/extract_crypto_ccxt.py
import ccxt
import pandas as pd
import time
from datetime import datetime, timezone
import os

def extract_binance_crypto(symbol='BTC/USDT', timeframe='15m', dal_anno=2024):
    print(f"🔄 Inizio estrazione {symbol} ({timeframe}) dal {dal_anno} via Binance CCXT...")
    exchange = ccxt.binance({
        'enableRateLimit': True,
        'options': {'defaultType': 'spot'}
    })
    
    since = exchange.parse8601(f'{dal_anno}-01-01T00:00:00Z')
    limit = 1000
    all_ohlcv = []
    
    while True:
        try:
            ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit)
            if not ohlcv:
                break
            
            all_ohlcv.extend(ohlcv)
            since = ohlcv[-1][0] + 1
            
            current_date = datetime.fromtimestamp(since / 1000.0, tz=timezone.utc).strftime('%Y-%m-%d')
            print(f"Scaricato blocco fino al {current_date}... (Totale candele: {len(all_ohlcv)})")
            time.sleep(exchange.rateLimit / 1000)
            
        except ccxt.NetworkError as e:
            print(f"⚠️ Errore di rete: {e}. Ritento tra 5 secondi...")
            time.sleep(5)
        except ccxt.ExchangeError as e:
            print(f"❌ Errore Exchange: {e}")
            break
            
    df = pd.DataFrame(all_ohlcv, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='ms')
    df.drop(columns=['Timestamp'], inplace=True)
    
    df.drop_duplicates(subset=['Datetime'], inplace=True)
    df.sort_values('Datetime', inplace=True)
    
    output_dir = os.path.dirname(os.path.abspath(__file__))
    output_file = os.path.join(output_dir, "raw_crypto_M15.csv")
    df.to_csv(output_file, index=False)
    print(f"✅ Estrazione Crypto completata: {len(df)} candele salvate in {output_file}.")

if __name__ == "__main__":
    extract_binance_crypto()


In [ ]:
%%writefile /content/drive/MyDrive/alfacore_v8/data_pipeline/extract_capital_rest.py
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("CAPITAL_API_KEY")
EMAIL = os.getenv("CAPITAL_EMAIL")
PASSWORD = os.getenv("CAPITAL_PASSWORD")

def login_capital():
    url = "https://demo-api-capital.backend-capital.com/api/v1/session"
    payload = {"identifier": EMAIL, "password": PASSWORD}
    headers = {"X-CAP-API-KEY": API_KEY, "Content-Type": "application/json"}
    
    res = requests.post(url, json=payload, headers=headers)
    if res.status_code != 200:
        raise Exception(f"Errore Login: {res.text}")
        
    return res.headers.get("CST"), res.headers.get("X-SECURITY-TOKEN")

def extract_capital_traditional(epic="US100", data_inizio="2024-01-01T00:00:00"):
    if not API_KEY or not EMAIL or not PASSWORD:
        raise ValueError("Credenziali mancanti nel file .env")

    print("🔑 Autenticazione in corso su Capital.com API...")
    cst, x_sec = login_capital()
    headers = {"CST": cst, "X-SECURITY-TOKEN": x_sec, "X-CAP-API-KEY": API_KEY}
    
    API_URL = "https://demo-api-capital.backend-capital.com"
    data_target = datetime.strptime(data_inizio, "%Y-%m-%dT%H:%M:%S")
    current_end_time = datetime.utcnow()
    tutti_i_dati = []
    
    print(f"🔄 Inizio estrazione {epic} (M1) a ritroso fino al {data_inizio}...")
    
    while current_end_time > data_target:
        str_end_time = current_end_time.strftime("%Y-%m-%dT%H:%M:%S")
        url = f"{API_URL}/api/v1/prices/{epic}"
        params = {"resolution": "MINUTE_1", "max": 1000, "to": str_end_time}
        
        try:
            res = requests.get(url, headers=headers, params=params)
            if res.status_code != 200:
                print(f"⚠️ Errore API: {res.text}. Ritento...")
                time.sleep(2)
                continue
                
            data = res.json()
            prices = data.get("prices", [])
            
            if not prices:
                print("🛑 Nessun altro dato disponibile dal broker.")
                break
                
            for p in prices:
                tutti_i_dati.append({
                    "Datetime": pd.to_datetime(p["snapshotTime"]),
                    "Open": p["openPrice"]["ask"],
                    "High": p["highPrice"]["ask"],
                    "Low": p["lowPrice"]["ask"],
                    "Close": p["closePrice"]["ask"],
                    "Volume": p["lastTradedVolume"]
                })
                
            oldest_time = pd.to_datetime(prices[0]["snapshotTime"])
            current_end_time = oldest_time - timedelta(seconds=1)
            
            print(f"Scaricato blocco. Data più antica: {oldest_time}. (Tot: {len(tutti_i_dati)})")
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Errore fatale di rete: {e}")
            break
            
    if not tutti_i_dati:
        return
        
    df = pd.DataFrame(tutti_i_dati)
    df.drop_duplicates(subset=['Datetime'], inplace=True)
    df.sort_values('Datetime', inplace=True)
    
    output_dir = os.path.dirname(os.path.abspath(__file__))
    output_file = os.path.join(output_dir, "raw_trad_M1.csv")
    df.to_csv(output_file, index=False)
    print(f"✅ Estrazione Tradizionale completata: {len(df)} candele salvate in {output_file}.")

if __name__ == "__main__":
    extract_capital_traditional()


In [ ]:
%%writefile /content/drive/MyDrive/alfacore_v8/data_pipeline/build_nlp_oracles.py
import pandas as pd
import numpy as np
import os
import torch
from transformers import pipeline

def build_nlp_oracles(csv_path="news_dump.csv", is_crypto=True, batch_size=256):
    print("🧠 Inizializzazione Oracolo NLP (HuggingFace)...")
    device = 0 if torch.cuda.is_available() else -1
    model_name = "ElfaI/finbert-sentiment" if not is_crypto else "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    
    nlp_pipe = pipeline("sentiment-analysis", model=model_name, device=device)
    
    if not os.path.exists(csv_path):
        dates = pd.date_range(start='2024-01-01', end='2024-06-01', freq='2H')
        df_news = pd.DataFrame({'Datetime': dates, 'Text': ["Market is surging!"] * len(dates)})
    else:
        df_news = pd.read_csv(csv_path)
        df_news['Datetime'] = pd.to_datetime(df_news['Datetime'])
        
    texts = df_news['Text'].astype(str).tolist()
    raw_scores = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = nlp_pipe(batch)
        for res in results:
            score = res['score'] if res['label'] in ['POSITIVE', 'positive', 'bullish'] else -res['score']
            raw_scores.append(score)
            
    df_news['Raw_BERT_Score'] = raw_scores
    df_news.set_index('Datetime', inplace=True)
    
    timeframe = '15min' if is_crypto else '1min'
    df_news_agg = df_news.resample(timeframe).agg(
        Raw_Shock=('Raw_BERT_Score', 'sum'),
        News_Volume=('Raw_BERT_Score', 'count')
    ).reset_index()
    
    df_news_agg.fillna(0, inplace=True)
    df_news_agg['BERT_Sentiment_EMA'] = df_news_agg['Raw_Shock'].ewm(span=14, adjust=False).mean()
    df_news_agg['Bars_Since_News'] = df_news_agg.groupby(df_news_agg['Raw_Shock'] != 0).cumcount()
    
    max_bars = 16 if is_crypto else 240
    df_news_agg['Time_Since_News_Scaled'] = np.exp(-df_news_agg['Bars_Since_News'] / (max_bars / 2))
    
    output_dir = os.path.dirname(os.path.abspath(__file__))
    output_name = "oracle_crypto_nlp.csv" if is_crypto else "oracle_trad_nlp.csv"
    output_file = os.path.join(output_dir, output_name)
    
    df_news_agg.to_csv(output_file, index=False)
    print(f"✅ Oracolo NLP salvato in {output_file}.")

if __name__ == "__main__":
    build_nlp_oracles(is_crypto=True)


In [ ]:
%%writefile /content/drive/MyDrive/alfacore_v8/data_pipeline/build_xgboost_oracles.py
import pandas as pd
import numpy as np
import xgboost as xgb
import os
from dateutil.relativedelta import relativedelta

def build_xgboost_oracles(price_csv="raw_crypto_M15.csv"):
    print("🌲 Inizializzazione Oracolo XGBoost (Walk-Forward Validation)...")
    if not os.path.exists(price_csv):
        dates = pd.date_range(start='2024-01-01', end='2024-06-01', freq='15T')
        df = pd.DataFrame({'Datetime': dates, 'Close': np.random.uniform(60000, 70000, len(dates))})
    else:
        df = pd.read_csv(price_csv)
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        
    df.sort_values('Datetime', inplace=True)
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['Target'] = (df['Log_Return'].shift(-1) > 0).astype(int)
    df.dropna(inplace=True)
    
    features = ['Log_Return'] 
    start_date = df['Datetime'].min()
    end_date = df['Datetime'].max()
    
    current_train_start = start_date
    df['XGB_Prob'] = np.nan
    
    model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=4,
        tree_method='hist', device='cuda' if xgb.core.core.xgb_build_info().get('USE_CUDA', False) else 'cpu'
    )
    
    while True:
        train_end = current_train_start + relativedelta(months=3)
        test_end = train_end + relativedelta(months=1)
        if train_end >= end_date: break
            
        train_mask = (df['Datetime'] >= current_train_start) & (df['Datetime'] < train_end)
        test_mask = (df['Datetime'] >= train_end) & (df['Datetime'] < test_end)
        
        df_train, df_test = df[train_mask], df[test_mask]
        if len(df_train) < 100 or len(df_test) < 10:
            current_train_start += relativedelta(months=1)
            continue
            
        model.fit(df_train[features], df_train['Target'])
        probs = model.predict_proba(df_test[features])[:, 1]
        df.loc[test_mask, 'XGB_Prob'] = probs
        current_train_start += relativedelta(months=1)
        
    df.dropna(subset=['XGB_Prob'], inplace=True)
    output_dir = os.path.dirname(os.path.abspath(__file__))
    output_file = os.path.join(output_dir, "oracle_xgboost_probs.csv")
    df[['Datetime', 'XGB_Prob']].to_csv(output_file, index=False)
    print(f"✅ Walk-Forward XGBoost Completato. Output: {output_file}.")

if __name__ == "__main__":
    build_xgboost_oracles()


In [ ]:
%%writefile /content/drive/MyDrive/alfacore_v8/data_pipeline/merge_master_datasets.py
import pandas as pd
import os

def merge_master_datasets(is_crypto=True):
    base_dir = os.path.dirname(os.path.abspath(__file__))
    file_prezzi = "raw_crypto_M15.csv" if is_crypto else "raw_trad_M1.csv"
    file_nlp = "oracle_crypto_nlp.csv" if is_crypto else "oracle_trad_nlp.csv"
    file_xgb = "oracle_xgboost_probs.csv" 
    
    percorso_prezzi = os.path.join(base_dir, file_prezzi)
    percorso_nlp = os.path.join(base_dir, file_nlp)
    percorso_xgb = os.path.join(base_dir, file_xgb)
    
    if not os.path.exists(percorso_prezzi): return
    df_prezzi = pd.read_csv(percorso_prezzi)
    df_prezzi['Datetime'] = pd.to_datetime(df_prezzi['Datetime'])
    df_prezzi.sort_values('Datetime', inplace=True)
    
    if os.path.exists(percorso_nlp):
        df_nlp = pd.read_csv(percorso_nlp)
        df_nlp['Datetime'] = pd.to_datetime(df_nlp['Datetime'])
        df_nlp.sort_values('Datetime', inplace=True)
        df_master = pd.merge_asof(df_prezzi, df_nlp, on='Datetime', direction='backward')
        df_master.fillna({'BERT_Sentiment_EMA': 0.0, 'Time_Since_News_Scaled': 0.0}, inplace=True)
    else:
        df_master = df_prezzi.copy()
        df_master['BERT_Sentiment_EMA'] = 0.0
        df_master['Time_Since_News_Scaled'] = 0.0
        
    if os.path.exists(percorso_xgb):
        df_xgb = pd.read_csv(percorso_xgb)
        df_xgb['Datetime'] = pd.to_datetime(df_xgb['Datetime'])
        df_xgb.sort_values('Datetime', inplace=True)
        df_master = pd.merge_asof(df_master, df_xgb, on='Datetime', direction='backward')
        df_master.dropna(subset=['XGB_Prob'], inplace=True)
    else:
        df_master['XGB_Prob'] = 0.5
        
    df_master['Log_Return'] = df_master['Close'] / df_master['Close'].shift(1) - 1
    df_master['TR'] = df_master['High'] - df_master['Low']
    atr = df_master['TR'].rolling(14).mean()
    df_master['ATR_Z_Score'] = (atr - atr.rolling(50).mean()) / atr.rolling(50).std()
    df_master['Mom_50'] = df_master['Close'] / df_master['Close'].shift(50) - 1
    
    df_master.dropna(inplace=True)
    output_name = "Master_Crypto_V8.csv" if is_crypto else "Master_Trad_V8.csv"
    output_file = os.path.join(base_dir, output_name)
    df_master.to_csv(output_file, index=False)
    print(f"✅ FASE 1 COMPLETATA. Generato: {output_file}")

if __name__ == "__main__":
    merge_master_datasets(is_crypto=True)
